# Taller 05: Clustering DBSCAN

**Asignatura:** Inteligencia Artificial  
**Universidad del Norte**  
**Programa de Ingeniería de Sistemas**

### Objetivo

Diseñar e implementar un algoritmo de clustering DBSCAN aplicado a un conjunto de datos sintéticos bidimensionales que contenga al menos 20 grupos amorfos y datos atípicos (outliers).

En este taller se realizará:

- Generación de datos sintéticos en dos dimensiones.
- Generación de al menos 20 grupos amorfos.
- Generación de datos atípicos distribuidos uniformemente.
- Análisis exploratorio mediante gráficos de k-distancia.
- Selección de los parámetros `eps` y `MinPts`.
- Implementación del algoritmo DBSCAN.
- Evaluación del clustering mediante dos métricas.
- Análisis y discusión de los resultados.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, davies_bouldin_score

np.random.seed(42)

print("Librerías importadas correctamente.")

## 1. Fundamentos de DBSCAN

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) es un algoritmo de clustering basado en densidad.

A diferencia de algoritmos como K-Means, DBSCAN no necesita conocer previamente el número de grupos que existen en los datos.

El algoritmo utiliza principalmente dos parámetros:

- **eps:** distancia máxima que se considera para determinar si dos puntos pertenecen a la misma vecindad.
- **MinPts:** número mínimo de puntos necesarios dentro de una vecindad para considerar que existe una región densa.

DBSCAN clasifica los puntos principalmente en:

- **Punto núcleo:** tiene al menos `MinPts` puntos dentro de una distancia `eps`.
- **Punto frontera:** no tiene suficientes vecinos para ser núcleo, pero pertenece a la vecindad de un punto núcleo.
- **Punto ruido:** no pertenece a ningún grupo y no es alcanzado desde un punto núcleo.

Una de las principales ventajas de DBSCAN es que puede encontrar grupos con formas arbitrarias y detectar automáticamente puntos considerados ruido.

In [ ]:
def generar_grupo_amorfo(centro, n=150, escala=1.0, tipo=0):
    """
    Genera un grupo de puntos con diferentes formas amorfas.
    """

    t = np.random.uniform(0, 2 * np.pi, n)
    
    if tipo == 0:
        # Forma ondulada
        r = 0.6 + 0.25 * np.sin(3 * t)
        x = r * np.cos(t)
        y = r * np.sin(t)

    elif tipo == 1:
        # Forma alargada y curva
        x = 1.2 * np.cos(t)
        y = 0.45 * np.sin(t) + 0.25 * np.sin(3 * t)

    elif tipo == 2:
        # Forma irregular
        r = 0.5 + 0.35 * np.sin(2 * t + 1)
        x = r * np.cos(t)
        y = r * np.sin(t)

    elif tipo == 3:
        # Forma de arco
        x = 0.9 * np.cos(t)
        y = 0.35 * np.sin(t)

    else:
        # Forma elíptica irregular
        x = 0.8 * np.cos(t)
        y = 0.5 * np.sin(t)

    puntos = np.column_stack((x, y))

    # Agregar variación aleatoria
    puntos += np.random.normal(0, 0.12, puntos.shape)

    # Escalar
    puntos *= escala

    # Trasladar al centro
    puntos += np.array(centro)

    return puntos

In [ ]:
# Centros de los 20 grupos
centros = [
    (-18, -10),
    (-12, -10),
    (-6, -10),
    (0, -10),
    (6, -10),

    (-18, -3),
    (-12, -3),
    (-6, -3),
    (0, -3),
    (6, -3),

    (-18, 4),
    (-12, 4),
    (-6, 4),
    (0, 4),
    (6, 4),

    (-18, 11),
    (-12, 11),
    (-6, 11),
    (0, 11),
    (6, 11)
]

grupos = []

for i, centro in enumerate(centros):
    grupo = generar_grupo_amorfo(
        centro=centro,
        n=150,
        escala=1.8,
        tipo=i % 5
    )
    
    grupos.append(grupo)

X = np.vstack(grupos)

print("Cantidad total de puntos:", len(X))
print("Cantidad de grupos generados:", len(grupos))

In [ ]:
# Generar outliers uniformemente distribuidos
n_outliers = 150

outliers_x = np.random.uniform(-23, 11, n_outliers)
outliers_y = np.random.uniform(-15, 16, n_outliers)

outliers = np.column_stack((outliers_x, outliers_y))

# Combinar grupos y outliers
X_final = np.vstack((X, outliers))

print("Puntos pertenecientes a grupos:", len(X))
print("Outliers:", len(outliers))
print("Total de puntos:", len(X_final))

In [ ]:
plt.figure(figsize=(12, 8))

plt.scatter(
    X_final[:, 0],
    X_final[:, 1],
    s=8
)

plt.title("Datos sintéticos generados")
plt.xlabel("X")
plt.ylabel("Y")
plt.grid(alpha=0.3)

plt.show()

## 2. Análisis de k-distancia

Para seleccionar un valor apropiado de `eps`, utilizaremos un gráfico de k-distancia.

La idea consiste en calcular, para cada punto, la distancia hasta su k-ésimo vecino más cercano.

Cuando las distancias se ordenan de menor a mayor, normalmente aparece una zona donde la distancia comienza a aumentar rápidamente. Este cambio puede utilizarse como referencia para seleccionar `eps`.

El valor de `k` está relacionado con `MinPts`.

En este caso analizaremos diferentes valores de `MinPts` y observaremos cómo cambia la distribución de las distancias.

In [ ]:
def calcular_k_distancia(datos, k):
    vecinos = NearestNeighbors(n_neighbors=k)
    vecinos.fit(datos)

    distancias, _ = vecinos.kneighbors(datos)

    k_distancias = np.sort(distancias[:, -1])

    return k_distancias

In [ ]:
valores_k = [4, 5, 6, 8, 10]

plt.figure(figsize=(12, 7))

for k in valores_k:
    k_distancias = calcular_k_distancia(X_final, k)
    
    plt.plot(
        k_distancias,
        label=f"k = {k}"
    )

plt.title("Gráficos de k-distancia")
plt.xlabel("Puntos ordenados")
plt.ylabel("Distancia al k-ésimo vecino")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

## 3. Selección de MinPts

Para este conjunto de datos se utilizará inicialmente:

**MinPts = 5**

Este valor proporciona un equilibrio entre detectar regiones suficientemente densas y evitar que pequeñas concentraciones de puntos sean consideradas clusters.

Posteriormente se probarán diferentes valores de `MinPts` junto con diferentes valores de `eps` para comparar su comportamiento.

In [ ]:
min_pts = 5

k_distancias = calcular_k_distancia(X_final, min_pts)

plt.figure(figsize=(12, 7))

plt.plot(k_distancias)

plt.title(f"Gráfico de k-distancia para MinPts = {min_pts}")
plt.xlabel("Puntos ordenados")
plt.ylabel("Distancia al 5.º vecino")
plt.grid(alpha=0.3)

plt.show()

### Selección de eps

A partir del gráfico de k-distancia se busca identificar el punto donde comienza el incremento pronunciado de las distancias.

Para este conjunto de datos se utilizará inicialmente:

- **eps = 1.0**
- **MinPts = 5**

Estos valores no se consideran definitivos. Se realizará una experimentación con diferentes combinaciones para determinar cuál produce mejores resultados.

In [ ]:
def region_query(X, punto, eps):
    """
    Encuentra todos los puntos que están a una distancia
    menor o igual a eps del punto indicado.
    """
    
    distancias = np.linalg.norm(X - X[punto], axis=1)
    
    vecinos = np.where(distancias <= eps)[0]
    
    return vecinos


def expandir_cluster(X, etiquetas, punto, vecinos, cluster_id, eps, min_pts):
    """
    Expande un cluster a partir de un punto núcleo.
    """

    etiquetas[punto] = cluster_id

    i = 0

    while i < len(vecinos):

        vecino = vecinos[i]

        if etiquetas[vecino] == -2:
            etiquetas[vecino] = cluster_id

        elif etiquetas[vecino] == -1:
            etiquetas[vecino] = cluster_id

        nuevos_vecinos = region_query(X, vecino, eps)

        if len(nuevos_vecinos) >= min_pts:
            for nuevo in nuevos_vecinos:
                if nuevo not in vecinos:
                    vecinos = np.append(vecinos, nuevo)

        i += 1


def dbscan(X, eps, min_pts):
    """
    Implementación propia del algoritmo DBSCAN.

    etiquetas:
    -2 = punto no visitado
    -1 = ruido
    0, 1, 2, ... = cluster
    """

    etiquetas = np.full(len(X), -2)

    cluster_id = 0

    for punto in range(len(X)):

        if etiquetas[punto] != -2:
            continue

        vecinos = region_query(X, punto, eps)

        if len(vecinos) < min_pts:
            etiquetas[punto] = -1

        else:
            expandir_cluster(
                X,
                etiquetas,
                punto,
                vecinos,
                cluster_id,
                eps,
                min_pts
            )

            cluster_id += 1

    return etiquetas

In [ ]:
eps = 1.0
min_pts = 5

etiquetas = dbscan(
    X_final,
    eps,
    min_pts
)

n_clusters = len(set(etiquetas)) - (1 if -1 in etiquetas else 0)
n_ruido = np.sum(etiquetas == -1)

print("Parámetros utilizados:")
print("eps =", eps)
print("MinPts =", min_pts)
print()
print("Clusters encontrados:", n_clusters)
print("Puntos clasificados como ruido:", n_ruido)

In [ ]:
plt.figure(figsize=(12, 8))

clusters = sorted(set(etiquetas))

for cluster in clusters:

    puntos_cluster = X_final[etiquetas == cluster]

    if cluster == -1:
        plt.scatter(
            puntos_cluster[:, 0],
            puntos_cluster[:, 1],
            s=15,
            marker="x",
            label="Ruido"
        )
    else:
        plt.scatter(
            puntos_cluster[:, 0],
            puntos_cluster[:, 1],
            s=8,
            label=f"Cluster {cluster}"
        )

plt.title(
    f"DBSCAN - eps={eps}, MinPts={min_pts}"
)

plt.xlabel("X")
plt.ylabel("Y")

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    fontsize=8
)

plt.grid(alpha=0.3)
plt.show()

## 4. Experimentación con diferentes parámetros

DBSCAN es sensible a los parámetros `eps` y `MinPts`.

Por esta razón se probarán diferentes combinaciones de parámetros y se analizará:

- Número de clusters encontrados.
- Cantidad de puntos considerados ruido.
- Silhouette Score.
- Davies-Bouldin Index.

El objetivo es encontrar una configuración que permita obtener grupos coherentes y una cantidad razonable de puntos considerados ruido.

In [ ]:
resultados = []

valores_eps = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3]
valores_min_pts = [4, 5, 6, 8, 10]

for eps_test in valores_eps:

    for min_pts_test in valores_min_pts:

        etiquetas_test = dbscan(
            X_final,
            eps_test,
            min_pts_test
        )

        n_clusters_test = len(
            set(etiquetas_test)
        ) - (1 if -1 in etiquetas_test else 0)

        n_ruido_test = np.sum(
            etiquetas_test == -1
        )

        # Para calcular las métricas necesitamos al menos 2 clusters
        mascara = etiquetas_test != -1

        etiquetas_sin_ruido = etiquetas_test[mascara]
        datos_sin_ruido = X_final[mascara]

        if (
            n_clusters_test >= 2
            and len(set(etiquetas_sin_ruido)) >= 2
        ):

            silhouette = silhouette_score(
                datos_sin_ruido,
                etiquetas_sin_ruido
            )

            davies = davies_bouldin_score(
                datos_sin_ruido,
                etiquetas_sin_ruido
            )

        else:
            silhouette = np.nan
            davies = np.nan

        resultados.append({
            "eps": eps_test,
            "MinPts": min_pts_test,
            "Clusters": n_clusters_test,
            "Ruido": n_ruido_test,
            "Silhouette": silhouette,
            "Davies-Bouldin": davies
        })

resultados_df = pd.DataFrame(resultados)

resultados_df

In [ ]:
resultados_ordenados = resultados_df.sort_values(
    by="Silhouette",
    ascending=False
)

resultados_ordenados.head(10)

## 5. Selección de los parámetros finales

La selección final de parámetros se realizará considerando conjuntamente:

1. El análisis del gráfico de k-distancia.
2. El número de clusters encontrados.
3. La cantidad de puntos clasificados como ruido.
4. El valor del Silhouette Score.
5. El valor del índice Davies-Bouldin.

Un mayor valor de Silhouette indica una mejor separación y cohesión de los clusters.

Por otra parte, un menor valor de Davies-Bouldin indica clusters más compactos y mejor separados.

A partir de estos criterios se seleccionará la configuración final.

In [ ]:
mejor_resultado = resultados_ordenados.iloc[0]

eps_final = mejor_resultado["eps"]
min_pts_final = int(mejor_resultado["MinPts"])

print("Configuración seleccionada:")
print("eps =", eps_final)
print("MinPts =", min_pts_final)
print("Silhouette =", mejor_resultado["Silhouette"])
print("Davies-Bouldin =", mejor_resultado["Davies-Bouldin"])

In [ ]:
etiquetas_finales = dbscan(
    X_final,
    eps_final,
    min_pts_final
)

clusters_finales = len(
    set(etiquetas_finales)
) - (1 if -1 in etiquetas_finales else 0)

ruido_final = np.sum(
    etiquetas_finales == -1
)

print("RESULTADOS FINALES")
print("------------------")
print("eps:", eps_final)
print("MinPts:", min_pts_final)
print("Clusters encontrados:", clusters_finales)
print("Puntos de ruido:", ruido_final)

In [ ]:
plt.figure(figsize=(13, 9))

clusters = sorted(set(etiquetas_finales))

for cluster in clusters:

    puntos = X_final[etiquetas_finales == cluster]

    if cluster == -1:

        plt.scatter(
            puntos[:, 0],
            puntos[:, 1],
            s=20,
            marker="x",
            label="Ruido"
        )

    else:

        plt.scatter(
            puntos[:, 0],
            puntos[:, 1],
            s=10,
            label=f"Cluster {cluster}"
        )

plt.title(
    f"Resultado final de DBSCAN\n"
    f"eps={eps_final}, MinPts={min_pts_final}"
)

plt.xlabel("X")
plt.ylabel("Y")

plt.legend(
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    fontsize=8
)

plt.grid(alpha=0.3)

plt.show()

## 6. Evaluación del clustering

### Silhouette Score

El Silhouette Score permite evaluar qué tan bien asignados están los puntos a sus respectivos clusters.

Su valor se encuentra entre -1 y 1.

- Valores cercanos a **1** indican clusters bien separados y compactos.
- Valores cercanos a **0** indican que existen puntos ubicados entre diferentes clusters.
- Valores negativos pueden indicar asignaciones incorrectas.

Los puntos clasificados como ruido por DBSCAN (`-1`) se excluyen del cálculo, debido a que no pertenecen a ningún cluster.

In [ ]:
mascara_final = etiquetas_finales != -1

X_evaluacion = X_final[mascara_final]
etiquetas_evaluacion = etiquetas_finales[mascara_final]

silhouette_final = silhouette_score(
    X_evaluacion,
    etiquetas_evaluacion
)

print("Silhouette Score:", silhouette_final)

### Davies-Bouldin Index

El índice Davies-Bouldin mide la similitud entre los clusters considerando su dispersión interna y la distancia entre ellos.

En este caso:

- Un valor **menor** indica un mejor resultado.
- Un valor mayor indica que los clusters presentan mayor similitud o menor separación.

Al igual que en el cálculo anterior, los puntos clasificados como ruido son excluidos de la evaluación.

In [ ]:
davies_final = davies_bouldin_score(
    X_evaluacion,
    etiquetas_evaluacion
)

print("Davies-Bouldin Index:", davies_final)

In [ ]:
resumen = pd.DataFrame({
    "Parámetro": [
        "eps",
        "MinPts",
        "Número de clusters",
        "Puntos de ruido",
        "Silhouette Score",
        "Davies-Bouldin Index"
    ],
    
    "Resultado": [
        eps_final,
        min_pts_final,
        clusters_finales,
        ruido_final,
        silhouette_final,
        davies_final
    ]
})

resumen

## 7. Análisis de resultados

A partir de la ejecución del algoritmo DBSCAN se pueden analizar diferentes aspectos.

### Número de clusters

El algoritmo permitió identificar automáticamente diferentes agrupaciones en los datos sin especificar previamente el número de clusters.

Esto resulta especialmente útil para este problema porque los datos fueron generados con grupos de formas amorfas.

### Puntos de ruido

DBSCAN también identificó puntos que no pertenecían a regiones suficientemente densas y los clasificó como ruido.

Estos puntos corresponden principalmente a los datos atípicos generados uniformemente.

### Influencia de eps

El parámetro `eps` controla el tamaño de la vecindad utilizada por DBSCAN.

Un valor demasiado pequeño puede provocar que muchos puntos sean considerados ruido o que un mismo grupo se divida en varios clusters.

Un valor demasiado grande puede provocar que diferentes grupos sean unidos.

### Influencia de MinPts

`MinPts` determina la cantidad mínima de puntos necesaria para considerar que una región tiene suficiente densidad.

Valores pequeños pueden generar clusters a partir de pequeñas concentraciones de puntos, mientras que valores grandes hacen que el algoritmo sea más restrictivo.

### Evaluación

El Silhouette Score permite analizar la cohesión y separación de los clusters.

El índice Davies-Bouldin permite analizar la similitud entre los diferentes clusters.

Ambas métricas se utilizaron conjuntamente para analizar la calidad del agrupamiento.

## 8. Conclusiones

En este taller se implementó el algoritmo DBSCAN sobre un conjunto de datos sintéticos bidimensionales compuesto por más de 20 grupos con formas amorfas y datos atípicos distribuidos uniformemente.

El análisis mediante gráficos de k-distancia permitió establecer valores iniciales para los parámetros `eps` y `MinPts`. Posteriormente, se probaron diferentes combinaciones de estos parámetros para analizar su influencia sobre el resultado.

Una de las principales ventajas observadas de DBSCAN es su capacidad para identificar grupos con formas no necesariamente circulares o lineales, además de clasificar puntos aislados como ruido.

Finalmente, el clustering fue evaluado mediante Silhouette Score y Davies-Bouldin Index. Estas métricas permitieron analizar la calidad de los grupos obtenidos desde diferentes perspectivas.

Los resultados muestran que la selección adecuada de `eps` y `MinPts` es fundamental para obtener un agrupamiento coherente, especialmente cuando los datos contienen ruido y grupos con formas amorfas.